In [ ]:
"""
Goal: Fine-tune a sentence transformer model for Bengali Question Answering and implement a hybrid search system combining semantic and keyword-based search.
Key Steps:
Data Preparation: Created positive question-answer pairs for training.
Model Training: Fine-tuned a SentenceTransformer model using CosineSimilarityLoss.
Model Saving: Saved the fine-tuned model for later use.
Document Embedding: Encoded queries using the trained model and indexed embeddings using FAISS for fast retrieval.
TF-IDF Vectorization: Added keyword-based search using TF-IDF for matching query terms.
Hybrid Search: Combined semantic search (FAISS embeddings) and keyword-based search (TF-IDF) using a weighted scoring system.
Inference: Built a function to predict the most relevant document for a given query.
Outcome: A hybrid search system that ranks documents based on both semantic relevance and keyword matching for Bengali queries.

"QUERY": ["PREDICTED ANSWER", "ACTUAL ANSWER"]
----------------------------------------------
"এটেন্ডেন্স কত?": ["আজকের স্ট্রাইক রেট কত?", "আজকের মোট Attendance কত?"],
"উপস্থিতি": ["আজকের স্ট্রাইক রেট কত?", "আজকের মোট Attendance কত?"],
"What’s today’s strike rate?": ["নিউট্রিশন ডিসপ্লের Overall Compliance কত?", "আজকের স্ট্রাইক রেট কত?"],
"আজকের কমপ্লায়েন্স কত?": ["আজকের স্ট্রাইক রেট কত?", "আজকের ডিসপ্লে অনুযায়ী Overall Compliance কত?"],
"Nutrition display compliance koto?": ["আজকের UBL Sachet Share কত আউটলেট অনুযায়ী?", "নিউট্রিশন ডিসপ্লের Overall Compliance কত?"],
"আজকের nutrition display compliance কত?": ["আজকের UBL Sachet Share কত আউটলেট অনুযায়ী?", "নিউট্রিশন ডিসপ্লের Overall Compliance কত?"],
"nutrition display কমপ্লায়েন্স?": ["কতগুলো আউটলেটে Shelf Talker পাওয়া যায়নি?", "নিউট্রিশন ডিসপ্লের Overall Compliance কত?"],
"How many outlets have been covered so far?": ["আজকের ডিসপ্লে অনুযায়ী Overall Compliance কত?", "এখন পর্যন্ত মোট কতটি আউটলেট কভার হয়েছে?"],
"Total outlets covered": ["আজকের ডিসপ্লে অনুযায়ী Overall Compliance কত?", "এখন পর্যন্ত মোট কতটি আউটলেট কভার হয়েছে?"],
"outlets covered": ["আজকে কতটি আউটলেট Pass করেছে?", "এখন পর্যন্ত মোট কতটি আউটলেট কভার হয়েছে?"],
"outlets কভার": ["আজকের UBL Sachet Share কত আউটলেট অনুযায়ী?", "এখন পর্যন্ত মোট কতটি আউটলেট কভার হয়েছে?"],
"পিওএসএম install": ["আজকের স্ট্রাইক রেট কত?", "আজকে কতটি আউটলেটে কতগুলো POSM ইনস্টল করা হয়েছে?"],
"আউটলেট pass হওয়া": ["আজকের স্ট্রাইক রেট কত?", "আজকে কতটি আউটলেট Pass করেছে?"],
"আউটলেট ফেইল করেছে?": ["আজকের স্ট্রাইক রেট কত?", "কতটি আউটলেট Failed করেছে?"],
"আউটলেট পাশ করেছে": ["আজকের স্ট্রাইক রেট কত?", "আজকে কতটি আউটলেট Pass করেছে?"],
"outlet পাশ করেছে?": ["আজকের স্ট্রাইক রেট কত?", "আজকে কতটি আউটলেট Pass করেছে?"],
"outlet ফেইল করেছে?": ["আজকের স্ট্রাইক রেট কত?", "কতটি আউটলেট Failed করেছে?"],
"এসওএস": ["আজকের স্ট্রাইক রেট কত?", "এখন পর্যন্ত SOS (Share of Shelf) এর রেজাল্ট কী?"],
"What’s today’s program-based compliance result?": ["আজকের ডিসপ্লে অনুযায়ী Overall Compliance কত?", "আজকের প্রোগ্রামভিত্তিক কমপ্লায়েন্স রেজাল্ট কী?"],
"program compliance result": ["আজকের ডিসপ্লে অনুযায়ী Overall Compliance কত?", "আজকের প্রোগ্রামভিত্তিক কমপ্লায়েন্স রেজাল্ট কী?"],
"ইউবিএল Sachet শেয়ার": ["আজকের স্ট্রাইক রেট কত?", "আজকের UBL Sachet Share কত আউটলেট অনুযায়ী?"],
"Sachet এডিয়ারেন্স ভুল": ["আজকের স্ট্রাইক রেট কত?", "কতটি আউটলেটে Sachet Adherence ভুল আছে?"],
"share of Shelf এর রেজাল্ট": ["আজকের UBL Sachet Share কত আউটলেট অনুযায়ী?", "এখন পর্যন্ত SOS (Share of Shelf) এর রেজাল্ট কী?"],
"শেয়ার অফ শেলফ এর রেজাল্ট": ["আজকের স্ট্রাইক রেট কত?", "এখন পর্যন্ত SOS (Share of Shelf) এর রেজাল্ট কী?"],
"শেয়ার অফ শেলফ result": ["আজকের স্ট্রাইক রেট কত?", "এখন পর্যন্ত SOS (Share of Shelf) এর রেজাল্ট কী?"],
"প্লানোগ্রাম Adherence vul": ["কতটি আউটলেটে Sachet Adherence ভুল আছে?", "কতগুলো আউটলেটে Planogram Adherence ভুল আছে?"],
"প্লানোগ্রাম এডিয়ারেন্স vul": ["আজকের প্রোগ্রামভিত্তিক কমপ্লায়েন্স রেজাল্ট কী?", "কতগুলো আউটলেটে Planogram Adherence ভুল আছে?"],
"প্লানোগ্রাম এডিয়ারেন্স ভুল": ["আজকের প্রোগ্রামভিত্তিক কমপ্লায়েন্স রেজাল্ট কী?", "কতগুলো আউটলেটে Planogram Adherence ভুল আছে?"]

"""
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from transformers import TrainingArguments
import os
import numpy as np
import faiss
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Disable wandb logs
os.environ["WANDB_DISABLED"] = "true"

# Step 1: Prepare training arguments
training_args = TrainingArguments(
    output_dir="./output",
    report_to="none",  # Disables wandb, tensorboard, etc.
    logging_dir="./logs",  # Optional: Add logging for monitoring
    per_device_train_batch_size=8,  # Increased batch size for faster training
    num_train_epochs=5,  # Reduced epochs to avoid overfitting
    logging_steps=500  # Log every 500 steps
)

# Step 2: Prepare your training examples
train_examples = [
    # # Negative examples (label=0) - Unrelated query and document pairs
    # InputExample(texts=["আজকের মোট Attendance কত?", "SOS er current result holo: Fabsol — UBL 44664 (100%), Competitor 0 (0%)"], label=0),  # Unrelated
    # InputExample(texts=["আজকের স্ট্রাইক রেট কত?", "lux display er current status -- Planogram Adherence → Yes: 101, No: 79, Exclusivity → Yes: 102, No: 8, N/A: 70"], label=0),  # Unrelated
    # InputExample(texts=["আজকের ডিসপ্লে অনুযায়ী Overall Compliance কত?", "aj total 1825 outlet e planogram adherence vul ase."], label=0),  # Unrelated
    # InputExample(texts=["নিউট্রিশন ডিসপ্লের Overall Compliance কত?", "aj 7677 outlet e sachet adherence vul ase."], label=0),  # Unrelated
    # InputExample(texts=["এখন পর্যন্ত মোট কতটি আউটলেট কভার হয়েছে?", "oral care display er exclusivity 63% thik ase."], label=0),  # Unrelated
    # InputExample(texts=["আজকে কতটি আউটলেটে কতগুলো POSM ইনস্টল করা হয়েছে?", "SOS er current result holo: Fabsol — UBL 44664 (100%), Competitor 0 (0%)"], label=0),  # Unrelated
    # InputExample(texts=["আজকে কতটি আউটলেট Pass করেছে?", "lux display er current status -- Planogram Adherence → Yes: 101, No: 79, Exclusivity → Yes: 102, No: 8, N/A: 70"], label=0),  # Unrelated
    # InputExample(texts=["কতটি আউটলেট Failed করেছে? কেন Failed হয়েছে?", "ajker program bhittik compliance result holo -- QPDS: 60%, NS: 84%, BS: 89%."], label=0),  # Unrelated
    # InputExample(texts=["আজকের প্রোগ্রামভিত্তিক কমপ্লায়েন্স রেজাল্ট কী?", "aj total 2305 outlet failed hoise, reason holo je ei outlet gulor moddhe jekono ekta display fail hoise."], label=0),  # Unrelated

    # Negative examples (incorrect predictions, label=0)
    InputExample(texts=["এটেন্ডেন্স কত?", "আজকের স্ট্রাইক রেট কত?"], label=0),  # Incorrect pair
    InputExample(texts=["উপস্থিতি", "আজকের স্ট্রাইক রেট কত?"], label=0),  # Incorrect pair
    InputExample(texts=["What’s today’s strike rate?", "নিউট্রিশন ডিসপ্লের Overall Compliance কত?"], label=0),  # Incorrect pair
    InputExample(texts=["আজকের কমপ্লায়েন্স কত?", "আজকের স্ট্রাইক রেট কত?"], label=0),  # Incorrect pair
    InputExample(texts=["Nutrition display compliance koto?", "আজকের UBL Sachet Share কত আউটলেট অনুযায়ী?"], label=0),  # Incorrect pair
    InputExample(texts=["nutrition display কমপ্লায়েন্স?", "কতগুলো আউটলেটে Shelf Talker পাওয়া যায়নি?"], label=0),  # Incorrect pair
    InputExample(texts=["How many outlets have been covered so far?", "আজকের ডিসপ্লে অনুযায়ী Overall Compliance কত?"], label=0),  # Incorrect pair
    InputExample(texts=["Total outlets covered", "আজকের ডিসপ্লে অনুযায়ী Overall Compliance কত?"], label=0),  # Incorrect pair
    InputExample(texts=["outlets covered", "আজকে কতটি আউটলেটে কতগুলো POSM ইনস্টল করা হয়েছে?"], label=0),  # Incorrect pair
    InputExample(texts=["আউটলেট pass হওয়া", "আজকের স্ট্রাইক রেট কত?"], label=0),  # Incorrect pair
    InputExample(texts=["আউটলেট ফেইল করেছে?", "আজকের স্ট্রাইক রেট কত?"], label=0),  # Incorrect pair
    InputExample(texts=["এসওএস", "আজকের স্ট্রাইক রেট কত?"], label=0),  # Incorrect pair
    InputExample(texts=["What’s today’s program-based compliance result?", "আজকের ডিসপ্লে অনুযায়ী Overall Compliance কত?"], label=0),  # Incorrect pair
    InputExample(texts=["program compliance result", "আজকের ডিসপ্লে অনুযায়ী Overall Compliance কত?"], label=0),  # Incorrect pair
    InputExample(texts=["ইউবিএল Sachet শেয়ার", "আজকের স্ট্রাইক রেট কত?"], label=0),  # Incorrect pair
    InputExample(texts=["Sachet এডিয়ারেন্স ভুল", "আজকের স্ট্রাইক রেট কত?"], label=0),  # Incorrect pair
    InputExample(texts=["share of Shelf এর রেজাল্ট", "আজকের UBL Sachet Share কত আউটলেট অনুযায়ী?"], label=0),  # Incorrect pair
    InputExample(texts=["শেয়ার অফ শেলফ এর রেজাল্ট", "আজকের স্ট্রাইক রেট কত?"], label=0),  # Incorrect pair
    InputExample(texts=["প্লানোগ্রাম Adherence vul", "আজকের স্ট্রাইক রেট কত?"], label=0),  # Incorrect pair
    InputExample(texts=["প্লানোগ্রাম এডিয়ারেন্স vul", "আজকের প্রোগ্রামভিত্তিক কমপ্লায়েন্স রেজাল্ট কী?"], label=0),  # Incorrect pair

    # Positive examples (correct pair, label=1)
    InputExample(texts=["প্লানোগ্রাম এডিয়ারেন্স vul", "কতগুলো আউটলেটে Planogram Adherence ভুল আছে?"], label=1),  # Correct pair
    InputExample(texts=["এটেন্ডেন্স কত?", "আজকের মোট Attendance কত?"], label=1),  # Correct pair
    InputExample(texts=["উপস্থিতি", "আজকের মোট Attendance কত?"], label=1),  # Correct pair
    InputExample(texts=["What’s today’s strike rate?", "আজকের স্ট্রাইক রেট কত?"], label=1),  # Correct pair
    InputExample(texts=["আজকের কমপ্লায়েন্স কত?", "আজকের ডিসপ্লে অনুযায়ী Overall Compliance কত?"], label=1),  # Correct pair
    InputExample(texts=["Nutrition display compliance koto?", "নিউট্রিশন ডিসপ্লের Overall Compliance কত?"], label=1),  # Correct pair
    InputExample(texts=["nutrition display কমপ্লায়েন্স?", "নিউট্রিশন ডিসপ্লের Overall Compliance কত?"], label=1),  # Correct pair
    InputExample(texts=["How many outlets have been covered so far?", "এখন পর্যন্ত মোট কতটি আউটলেট কভার হয়েছে?"], label=1),  # Correct pair
    InputExample(texts=["Total outlets covered", "এখন পর্যন্ত মোট কতটি আউটলেট কভার হয়েছে?"], label=1),  # Correct pair
    InputExample(texts=["outlets covered", "এখন পর্যন্ত মোট কতটি আউটলেট কভার হয়েছে?"], label=1),  # Correct pair
    InputExample(texts=["আউটলেট pass হওয়া", "আজকে কতটি আউটলেট Pass করেছে?"], label=1),  # Correct pair
    InputExample(texts=["আউটলেট ফেইল করেছে?", "কতটি আউটলেট Failed করেছে?"], label=1),  # Correct pair
    InputExample(texts=["এসওএস", "এখন পর্যন্ত SOS (Share of Shelf) এর রেজাল্ট কী?"], label=1),  # Correct pair
    InputExample(texts=["What’s today’s program-based compliance result?", "আজকের প্রোগ্রামভিত্তিক কমপ্লায়েন্স রেজাল্ট কী?"], label=1),  # Correct pair
    InputExample(texts=["program compliance result", "আজকের প্রোগ্রামভিত্তিক কমপ্লায়েন্স রেজাল্ট কী?"], label=1),  # Correct pair
    InputExample(texts=["ইউবিএল Sachet শেয়ার", "আজকের UBL Sachet Share কত আউটলেট অনুযায়ী?"], label=1),  # Correct pair
    InputExample(texts=["Sachet এডিয়ারেন্স ভুল", "কতটি আউটলেটে Sachet Adherence ভুল আছে?"], label=1),  # Correct pair
    InputExample(texts=["share of Shelf এর রেজাল্ট", "এখন পর্যন্ত SOS (Share of Shelf) এর রেজাল্ট কী?"], label=1),  # Correct pair
    InputExample(texts=["শেয়ার অফ শেলফ এর রেজাল্ট", "এখন পর্যন্ত SOS (Share of Shelf) এর রেজাল্ট কী?"], label=1),  # Correct pair
    InputExample(texts=["প্লানোগ্রাম Adherence vul", "কতগুলো আউটলেটে Planogram Adherence ভুল আছে?"], label=1),  # Correct pair
    InputExample(texts=["প্লানোগ্রাম এডিয়ারেন্স vul", "কতগুলো আউটলেটে Planogram Adherence ভুল আছে?"], label=1),  # Correct pair

]

# Step 3: Set up DataLoader and Loss function
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)  # Increased batch size
model = SentenceTransformer('all-MiniLM-L6-v2')  # Using the 'all-MiniLM-L6-v2' model
train_loss = losses.CosineSimilarityLoss(model)

# Step 4: Training the model
model.fit(train_objectives=[(train_dataloader, train_loss)], epochs=500, show_progress_bar=True)

# Step 5: Save the trained model
model.save("fine-tuned-question")

# Step 6: Prepare the dataframe with queries (replace with your own DataFrame)
df = pd.DataFrame([
    {"query": "আজকের মোট Attendance কত?", "doc": "ajker attendance 1015."},
    {"query": "আজকের স্ট্রাইক রেট কত?", "doc": "ajker total strike rate 96%."},
    {"query": "আজকের ডিসপ্লে অনুযায়ী Overall Compliance কত?", "doc": "ajker display overall compliance: PONDS — 88.76%, ORAL CARE QPDS — 63.60%, LUX BODYWASH QPDS — 74.79%, Juniors Clean Corner — 56.87%, PREMIUM PORTFOLIO QPDS — 39.50%, Hair Care — 88.40%, NS Double Shelf 2.2 — 84.61%, NS Single Shelf 1.1 — 81.74%, GAL — 89.41%, WINTER LOTION QPDS — 59.43%, NS Double Shelf 2.1 — 88.51%, NS Single Shelf 1.2 — 80.61%."},
    {"query": "নিউট্রিশন ডিসপ্লের Overall Compliance কত?", "doc": "nutrition display er overall compliance: NS Double Shelf 2.2 — 84.61%, NS Single Shelf 1.1 — 81.74%, NS Double Shelf 2.1 — 88.51%, NS Single Shelf 1.2 — 80.61%."},
    {"query": "এখন পর্যন্ত মোট কতটি আউটলেট কভার হয়েছে?", "doc": "ekhon porjonto 13281 ta outlet cover hoise."},
    {"query": "আজকে কতটি আউটলেটে কতগুলো POSM ইনস্টল করা হয়েছে?", "doc": "aj 13280 ta outlet e total 27540 POSM install kora hoise."},
    {"query": "আজকে কতটি আউটলেট Pass করেছে?", "doc": "aj total 2152 outlet pass hoise."},
    {"query": "কতটি আউটলেট Failed করেছে? কেন Failed হয়েছে?", "doc": "total 2305 outlet failed hoise, reason holo je ei outlet gulor moddhe jekono ekta display fail hoise."},
    {"query": "আজকের প্রোগ্রামভিত্তিক কমপ্লায়েন্স রেজাল্ট কী?", "doc": "ajker program bhittik compliance result holo -- QPDS: 60%, NS: 84%, BS: 89%."},
    {"query": "আজকের UBL Sachet Share কত আউটলেট অনুযায়ী?", "doc": "ajker outlet wise sachet share 86%."},
    {"query": "কতটি আউটলেটে Sachet Adherence ভুল আছে?", "doc": "aj 7677 outlet e sachet adherence vul ase."},
    {"query": "এখন পর্যন্ত SOS (Share of Shelf) এর রেজাল্ট কী?", "doc": "SOS er current result holo: Fabsol — UBL 44664 (100%), Competitor 0 (0%), Hair Care — UBL 68325 (100%), Competitor 0 (0%), Home & Hygiene — UBL 35953 (99%), Competitor 393 (1%), Nutrition — UBL 42305 (100%), Competitor 30 (0%), Oral Care — UBL 132743 (82%), Competitor 28476 (18%), Skin Care — UBL 184595 (100%), Competitor 864 (0%), Skin Cleansing — UBL 211454 (84%), Competitor 38812 (16%)."},
    {"query": "কতগুলো আউটলেটে Planogram Adherence ভুল আছে?", "doc": "aj total 1825 outlet e planogram adherence vul ase."},
    {"query": "কতগুলো আউটলেটে Shelf Talker পাওয়া যায়নি?", "doc": "aj 2324 outlet e kono shelf talker paoa jai nai."},
    {"query": "Lux ডিসপ্লের বর্তমান অবস্থা কী?", "doc": "lux display er current status -- Planogram Adherence → Yes: 101, No: 79, Exclusivity → Yes: 102, No: 8, N/A: 70, Compliance Avg → 74.79%, Variant Compliance Avg → 68.16%, Shelf Talker → Yes: 110, No: 70."},
    {"query": "Oral Care Display-এর Exclusivity Performance কী?", "doc": "oral care display er exclusivity 63% thik ase."}
])


# Step 7: Generate embeddings and create the FAISS index for similarity search
doc_embeddings = model.encode(df['query'].tolist(), show_progress_bar=True)

# Create the base index (flat L2 index)
base_index = faiss.IndexFlatL2(doc_embeddings.shape[1])

# Create the IVF index using the base index
index = faiss.IndexIVFFlat(base_index, doc_embeddings.shape[1], 10, faiss.METRIC_L2)  # 10 is the number of clusters
index.train(doc_embeddings)  # Train the index on the document embeddings
index.add(doc_embeddings)

# Save the FAISS index to a file
faiss.write_index(index, 'fine-tuned-question.index')
print("FAISS index saved successfully!")

# Step 8: Create TF-IDF vectorizer and calculate scores
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['query'].tolist())

# Step 9: Function to get TF-IDF score for a query
def get_tfidf_score(query):
    query_tfidf = vectorizer.transform([query])
    return tfidf_matrix.dot(query_tfidf.T).toarray().flatten()

# Step 10: Hybrid Search function (combines both semantic and keyword matching)
def hybrid_search(query, alpha=0.5, top_k=1):
    query_embedding = model.encode([query])  # Get the embedding for the query
    D, I = index.search(np.array(query_embedding), k=top_k)  # Perform FAISS search

    # Compute semantic scores from FAISS (1 - L2 distance)
    sem_scores = [1 - D[0][i] for i in range(top_k)]
    
    # Compute keyword-based TF-IDF scores
    keyword_scores = get_tfidf_score(query)

    # Combine the two scores (semantic + keyword)
    results = []
    for rank, idx in enumerate(I[0]):
        final_score = alpha * sem_scores[rank] + (1 - alpha) * keyword_scores[idx]
        results.append((df.iloc[idx]['query'], final_score))

    # Return sorted results based on the final score
    return sorted(results, key=lambda x: -x[1])

# Step 11: Inference with fine-tuned model
def predict(query):
    return hybrid_search(query, alpha=0.5, top_k=1)  # Return top 3 results for diversity

# Test the system
query = "program-based কমপ্লায়েন্স"
results = predict(query)
print(results)